# Practice Lab: Multimodal RAG (Lesson 8.9)

Module 8 · 8 exercises · Image-as-text + ColPali + cross-modal + audio/video + Indian OCR

Exercises 1, 4, 7, 8 run locally (pure Python).


# Lesson 8.9: Multimodal RAG — Beyond Text

8 exercises: 2E + 3M + 3C

Exercises 1, 4, 7, 8 run **locally** (pure Python). Ex 2, 3, 5, 6 are architecture/design.


In [ ]:
import numpy as np
import hashlib, re


---
## Exercise 1 (Easy): Image-as-Text RAG


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import hashlib
import numpy as np

def fe(t, dim=64):
    h = hashlib.sha256(t.lower().encode()).digest()
    v = np.array([b/255.0 for b in h[:dim]])
    for w, i in {"revenue":0,"q4":1,"chart":2,"growth":3,"course":4,"price":5,"14999":6,"cheapest":7,"refund":8,"student":9,"2025":10,"table":11,"image":12,"5000":14}.items():
        if w in t.lower() and i < dim: v[i] += 0.5
    return v / np.linalg.norm(v)

def cos(a, b): return np.dot(a, b)/(np.linalg.norm(a)*np.linalg.norm(b))

entries = [
    {"type":"text","src":"about.txt","text":"Netsetos edtech Hyderabad AI ML courses"},
    {"type":"text","src":"refund.txt","text":"Refund policy full refund 7 days 50 percent 8-30"},
    {"type":"image","src":"revenue_chart.png","text":"Bar chart quarterly revenue Q1 45 Q2 72 Q3 98 Q4 135 lakhs growth trend"},
    {"type":"image","src":"student_growth.png","text":"Bar chart student enrollment 2024 2000 2025 5000 students increase"},
    {"type":"table","src":"pricing_table","text":"Course Price Hours Rating GenAI 14999 146 4.8 Data Science 12999 120 Cloud 11999 110 Full-Stack 9999 80"},
]
for e in entries: e["emb"] = fe(e["text"])

queries = [("Q4 revenue?","image"),("Cheapest course?","table"),("Refund policy?","text"),
           ("Students in 2025?","image"),("Course pricing?","table")]

print("Multimodal RAG (text + image + table):")
correct = 0
for q, exp in queries:
    qe = fe(q)
    best = max(range(len(entries)), key=lambda i: cos(qe, entries[i]["emb"]))
    e = entries[best]; match = e["type"] == exp
    if match: correct += 1
    print(f"  Q: {q} -> [{e['type'].upper()}] {e['src']} {'OK' if match else 'MISS'}")

print(f"\nAccuracy: {correct}/{len(queries)} ({correct/len(queries):.0%})")
print(f"Strategy A: describe+embed (fast). B: +pass image (accurate)")
print(f"All modalities in ONE vector space as text embeddings")


</details>


---
## Exercise 4 (Medium): Multimodal Embeddings


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import hashlib
import numpy as np

def fe(t, dim=64):
    h = hashlib.sha256(t.lower().encode()).digest()
    v = np.array([b/255.0 for b in h[:dim]])
    for w, i in {"revenue":0,"chart":1,"architecture":5,"diagram":6,"pricing":8,"growth":9,"student":10}.items():
        if w in t.lower() and i < dim: v[i] += 0.5
    return v / np.linalg.norm(v)

def cos(a, b): return np.dot(a, b)/(np.linalg.norm(a)*np.linalg.norm(b))

entries = [
    {"type":"text","desc":"Course pricing table","emb":fe("pricing table course cost comparison")},
    {"type":"image","desc":"Revenue bar chart Q1-Q4","emb":fe("revenue bar chart quarterly growth")},
    {"type":"image","desc":"Architecture diagram RAG","emb":fe("architecture diagram RAG pipeline system")},
    {"type":"image","desc":"Student growth chart","emb":fe("student growth chart increase enrollment")},
]

print("Cross-Modal Retrieval:")
for q in ["Revenue charts","Architecture diagram","Pricing table","Student growth"]:
    qe = fe(q)
    best = max(range(len(entries)), key=lambda i: cos(qe, entries[i]["emb"]))
    e = entries[best]
    print(f"  Q: {q} -> [{e['type'].upper()}] {e['desc']} ({cos(qe, e['emb']):.3f})")

print(f"\n3 strategies: separate indices, unified (Nomic/BGE-VL), text grounding")
print(f"Nomic advantage: add image search WITHOUT re-indexing text")
print(f"Models: SigLIP 2, Nomic Vision, BGE-VL, CLIP/OpenCLIP")


</details>


---
## Exercise 7 (Challenge): Production Multimodal Router


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import hashlib

class Router:
    def __init__(self):
        self.cache = {}
        self.stats = {"text":0,"table":0,"image":0,"audio":0,"video":0,"cache_hits":0,"cost":0.0}
    def classify(self, desc):
        d = desc.lower()
        if any(k in d for k in ["chart","diagram","photo","image","png"]): return "image"
        if any(k in d for k in ["table","grid","spreadsheet","csv"]): return "table"
        if any(k in d for k in ["audio","mp3","wav","podcast"]): return "audio"
        if any(k in d for k in ["video","mp4","lecture","recording"]): return "video"
        return "text"
    def process(self, desc):
        ch = hashlib.md5(desc.encode()).hexdigest()[:12]
        if ch in self.cache:
            self.stats["cache_hits"] += 1
            return {**self.cache[ch], "cached": True}
        mod = self.classify(desc); self.stats[mod] += 1
        costs = {"text":0.005,"table":0.02,"image":0.04,"audio":0.02,"video":0.07}
        latency = {"text":15,"table":200,"image":2000,"audio":500,"video":5000}
        self.stats["cost"] += costs[mod]
        r = {"modality":mod,"cost":costs[mod],"latency_ms":latency[mod],"cached":False}
        self.cache[ch] = r; return r

router = Router()
docs = ["Annual report text paragraph","Revenue bar chart image","Pricing table CSV",
        "Architecture diagram PNG","Podcast MP3 episode","Training video MP4",
        "Refund policy text","Student enrollment table","Revenue bar chart image","Course spreadsheet CSV"]

print("Multimodal Router:")
for d in docs:
    r = router.process(d)
    c = " (CACHED)" if r["cached"] else ""
    print(f"  [{r['modality']:<6}] {d[:35]}... ${r['cost']:.3f} {r['latency_ms']}ms{c}")

print(f"\nStats: {router.stats}")
print(f"Cache: {router.stats['cache_hits']} hits (content-hash dedup)")
print(f"VLM description (1-5s) = dominant latency. Pre-process at ingestion.")


</details>


---
## Exercise 8 (Challenge): Indian Document OCR


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import re

print("Indian Document OCR:")
print(f"\nDevanagari OCR Comparison (illustrative single-study CERs - verify on your own docs):")
for name, hw, pr in [("Google Vision","14.6%","2-5%"),("PaddleOCR","~20%","3-8%"),("EasyOCR","46.8%","10-15%"),("Tesseract","85.5%","15-30%")]:
    print(f"  {name:<20} handwritten={hw:<8} printed={pr}")

print(f"\nID Card Regex Validation:")
tests = [("Aadhaar", r"\d{4}\s\d{4}\s\d{4}", ["1234 5678 9012","123 456 789","9876 5432 1098"]),
         ("PAN", r"[A-Z]{5}[0-9]{4}[A-Z]", ["ABCDE1234F","ABC12345F","ZZZZZ9999Z"])]
for doc, pattern, examples in tests:
    print(f"\n  {doc}: {pattern}")
    for ex in examples:
        valid = bool(re.search(pattern, ex))
        print(f"    '{ex}' -> {'VALID' if valid else 'INVALID'}")

print(f"\nDigiLocker: hundreds of millions of users, structured XML, zero OCR error")
print(f"Skip OCR for DigiLocker-enrolled users!")
print(f"Pipeline: preprocess -> classify -> OCR(hin+eng) -> LLM extract -> regex validate")

</details>


---
## Exercise 2 (Easy): ColPali Quickstart
Architecture/design. See practice-lab-8_9.html.


In [ ]:
# Exercise 2: ColPali Quickstart
pass


---
## Exercise 3 (Medium): Document AI Pipeline
Architecture/design. See practice-lab-8_9.html.


In [ ]:
# Exercise 3: Document AI Pipeline
pass


---
## Exercise 5 (Medium): Audio RAG with Whisper
Architecture/design. See practice-lab-8_9.html.


In [ ]:
# Exercise 5: Audio RAG with Whisper
pass


---
## Exercise 6 (Challenge): Video RAG Pipeline
Architecture/design. See practice-lab-8_9.html.


In [ ]:
# Exercise 6: Video RAG Pipeline
pass
